In [3]:
from dotenv import load_dotenv
load_dotenv()

from groq import Groq
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_react_agent, AgentExecutor
from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.agents import load_tools
from langchain_classic.chains.llm_math.prompt import PROMPT
from langchain_core.tools import tool
import numexpr
import wikipedia.wikipedia as wp_mod

wp_mod.USER_AGENT = "Assignment32Bot/1.0 (vichi.techsaga@gmail.com)"

## PART1: Tools in AI AGENTS
## TASK 1: Understanding Tools 

## Q.1: What is tool in an AI Agent ?
## Ans: In an AI Agent , A tool is an external function or capability that allows an AI Model to perform tasks beyond generating text.
## An LLM can understand questions and generate answers, but it cannot independently access external information or perform every action. Tools help it interact with the outside world.

## Q.2: Why do agents need tools ..?
## Ans: AI Agents need tools to perform actions and interact with the outside world, rather than just generating text using an LLM. An LLM can understand questions and generate answers, but it cannot independently access external information or perform actions unless it has the appropriate tools.
# Access real-time information: Search the internet for the latest information.
# Perform calculations: Use calculators or Python to solve mathematical problems.
# Access databases: Retrieve information from databases.
# Interact with APIs: Fetch data or perform operations through APIs.
# Automate tasks: Send emails, manage files, or perform other tasks.

## Q.3:  Difference bwtween a chatbot and an AI Agent
## Answer: A chatbot focuses on conversation, whereas an AI Agent focuses on decision-making and task execution.
# Chatbot: A chatbot mainly interacts with users by answering questions and providing information.
## AI Agent: An AI Agent can understand goals, make decisions, use tools, and perform actions to complete tasks.

In [4]:
## Task 2:  Built-in Tools in LangChain
## Initialize LLM
# llm = ChatGroq(
#     model = "openai/gpt-oss-120b",
#     temperature = 0
# )


llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# prompt = PROMPT.format(question="25 * 4 + 10")

## Initialize Tools

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression, e.g. '25 * 4 + 10'."""
    return str(numexpr.evaluate(expression.strip()))

# print(calculator.invoke({"expression": "25 * 4 + 10"}))  # 110

tools = load_tools(["llm-math", "wikipedia", "ddg-search"], llm=llm)
# print(tools)

print("Initialized tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:70]}...")


tool_map = {t.name.lower(): t for t in tools}
calculator = tool_map["calculator"]
wikipedia = tool_map["wikipedia"]
search = next(t for n, t in tool_map.items() if "search" in n)

# --- Calculator ---

print("\n---- Calculator Tool")
try:
    # llm-math takes a plain string, NOT a dict
    result = calculator.invoke("25 * 4 + 10")
    print(result)  # Answer: 110
except Exception as e:
    print(f"llm-math failed ({e}); falling back to numexpr")
    import numexpr
    print("Answer:", numexpr.evaluate("25 * 4 + 10"))



# --- Wikipedia ---
print("\n---- Wikipedia Tool")
try:
    result = wikipedia.invoke("LangChain")
    print(result[:500])
except Exception as e:
    print(f"Wikipedia error: {e}")


# --- Web Search ---
print("\n---- Web Search Tool (DuckDuckGo)")
try:
    result = search.invoke("What is LangChain?")
    print(result[:500])
except Exception as e:
    print(f"Search error: {e}")




Initialized tools:
  - Calculator: Useful for when you need to answer questions about math....
  - wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general...
  - duckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need to answer...

---- Calculator Tool
llm-math failed (Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 21h33m15.271742924s.', 'status': 

In [5]:
## Part 3: Creating Custom Tools and Toolkits
## Task 3: Create a Custom Tool


POLICIES = {
    "leave": "Employees get 24 paid leaves per year. Leave requests must be "
             "submitted at least 3 days in advance via the HR portal.",
    "work from home": "Employees may work from home up to 2 days per week "
                      "with manager approval.",
    "remote": "Employees may work from home up to 2 days per week "
              "with manager approval.",
    "working hours": "Standard working hours are 9:30 AM to 6:30 PM, "
                     "Monday to Friday, with a 1-hour lunch break.",
    "dress code": "Business casual Monday to Thursday. Casual wear is "
                  "allowed on Fridays.",
    "reimbursement": "Travel and business expenses must be submitted within "
                     "30 days with valid receipts.",
    "expense": "Travel and business expenses must be submitted within "
               "30 days with valid receipts.",
    "notice period": "The standard notice period for resignation is 60 days.",
    "resignation": "The standard notice period for resignation is 60 days.",
}


@tool 
def company_policy_lookup(query:str) -> str:
    """ Return the company policy information . 
    
    
    Use this when the user ask about company policies such as leave, work from home, working hours, dress code, reimbursement or notice period

    """


    q = query.lower().strip()

    for keyword, policy in POLICIES.items():
        if keyword in q:
            return policy
    available = ", ".join(sorted(set(POLICIES.keys())))      

    return f"No policy found for '{query}'. Available topics: {available}."  

    ## Test the tool independently

if __name__ == "__main__":
    # Tool metadata (what an LLM agent would see)
    print("Name       :", company_policy_lookup.name)
    print("Description:", company_policy_lookup.description)
    print("Args       :", company_policy_lookup.args)
 
    test_queries = [
        "What is the leave policy?",
        "Can I work from home?",
        "What are the working hours?",
        "Tell me about the dress code",
        "What is the notice period?",
        "What is the bonus policy?",   # not in data -> fallback message
    ]
 
    for q in test_queries:
        print(f"\n---- Query: {q}")
        print(company_policy_lookup.invoke(q))










Name       : company_policy_lookup
Description: Return the company policy information . 


   Use this when the user ask about company policies such as leave, work from home, working hours, dress code, reimbursement or notice period
Args       : {'query': {'title': 'Query', 'type': 'string'}}

---- Query: What is the leave policy?
Employees get 24 paid leaves per year. Leave requests must be submitted at least 3 days in advance via the HR portal.

---- Query: Can I work from home?
Employees may work from home up to 2 days per week with manager approval.

---- Query: What are the working hours?
Standard working hours are 9:30 AM to 6:30 PM, Monday to Friday, with a 1-hour lunch break.

---- Query: Tell me about the dress code
Business casual Monday to Thursday. Casual wear is allowed on Fridays.

---- Query: What is the notice period?
The standard notice period for resignation is 60 days.

---- Query: What is the bonus policy?
No policy found for 'What is the bonus policy?'. Available t